In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# Settings 
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

### 1. Load Preprocesssed Data (10 pts)  
* X_train shape: (768, 9)
* X_test shape: (154, 9)
* Number of numeric columns: 7
* Number of categorical columns: 2
* Which columns are features being made from:
    - glazing area
    - Surface to wall
    - Overall height and and surface area
    - log cooling load

In [2]:
df = pd.read_csv('data/raw/ee.csv')

X_train = pd.read_csv('data/processed/X_train.csv')
X_test = pd.read_csv('data/processed/X_test.csv')
y_train = pd.read_csv('data/processed/y_train.csv')
y_test = pd.read_csv('data/processed/y_test.csv')

print(f'X_train Loaded: {X_train.shape[0]:,} rows x {X_train.shape[1]} columns')
print(f'y_train Loaded: {y_train.shape[0]:,} rows x {y_train.shape[1]} columns')

print(f'X_test Loaded: {X_test.shape[0]:,} rows x {X_test.shape[1]} columns')
print(f'y_test Loaded: {y_test.shape[0]:,} rows x {y_test.shape[1]} columns')

X_train Loaded: 768 rows x 9 columns
y_train Loaded: 768 rows x 1 columns
X_test Loaded: 154 rows x 9 columns
y_test Loaded: 154 rows x 1 columns


In [3]:
X_train.head()

,relative_compactness,surface_area,wall_area,roof_area,overall_height,orientation,glazing_area,glazing_area_dist,cooling_load
0,0.98,514.5,294.0,110.25,7.0,2,0.0,0,21.33
1,0.98,514.5,294.0,110.25,7.0,3,0.0,0,21.33
2,0.98,514.5,294.0,110.25,7.0,4,0.0,0,21.33
3,0.98,514.5,294.0,110.25,7.0,5,0.0,0,21.33
4,0.90,563.5,318.5,122.50,7.0,2,0.0,0,28.28


In [4]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include='object').columns.tolist()
print(f'\nNumerical ({len(numerical_cols)}): {numerical_cols}')
print(f'Categorical ({len(categorical_cols)}): {categorical_cols}')

print("\nColumns:")
print(X_train.columns.tolist())

print("\nDtypes:")
print(X_train.dtypes.sort_index())

print("\nTarget distribution:")
print(y_train.value_counts(dropna=False))

(768, 9)
(154, 9)
(768, 1)
(154, 1)

Numerical (9): ['relative_compactness', 'surface_area', 'wall_area', 'roof_area', 'overall_height', 'orientation', 'glazing_area', 'glazing_area_dist', 'cooling_load']
Categorical (0): []

Columns:
['relative_compactness', 'surface_area', 'wall_area', 'roof_area', 'overall_height', 'orientation', 'glazing_area', 'glazing_area_dist', 'cooling_load']

Dtypes:
cooling_load            float64
glazing_area            float64
glazing_area_dist         int64
orientation               int64
overall_height          float64
relative_compactness    float64
roof_area               float64
surface_area            float64
wall_area               float64
dtype: object

Target distribution:
heating_load
15.16           6
13.00           5
10.68           4
32.31           4
15.55           4
               ..
7.18            1
6.05            1
41.30           1
41.32           1
10.38           1
Name: count, Length: 586, dtype: int64


### 2. Create New Features  
Create 4 new features
* Surface to wall ratio
* Overall height and surface area interaction
* Log cooling load
* Binary marker to indicate a building has a higher than average glazing area 

In [5]:
# Surface to wall ratio
X_train['surface_wall_ratio'] = X_train['surface_area'] / X_train['wall_area']
X_test['surface_wall_ratio'] = X_test['surface_area'] / X_test['wall_area']

# Check  
print(X_train['surface_wall_ratio'].describe())

count    768.000000
mean       2.141198
std        0.369576
min        1.588235
25%        1.835165
50%        2.100000
75%        2.413462
max        2.800000
Name: surface_wall_ratio, dtype: float64


In [6]:
# Height and Surface Area interaction
X_train['height_surface_interaction'] = X_train['overall_height'] * X_train['surface_area']
X_test['height_surface_interaction'] = X_test['overall_height'] * X_test['surface_area']

print(X_train['height_surface_interaction'].describe())

count     768.000000
mean     3394.270833
std       821.871001
min      2401.000000
25%      2636.812500
50%      3215.625000
75%      4158.875000
max      4630.500000
Name: height_surface_interaction, dtype: float64


In [7]:
# Log cooling load
X_train['log_cooling_load'] = np.log1p(X_train['cooling_load'])
X_test['log_cooling_load'] = np.log1p(X_test['cooling_load'])

print(X_train['log_cooling_load'].describe())

count    768.000000
mean       3.172162
std        0.375832
min        2.476538
25%        2.810606
50%        3.138966
75%        3.530250
max        3.892432
Name: log_cooling_load, dtype: float64


In [8]:
#Flag any Glazing Area above th median as High glazing
median_glazing = X_train['glazing_area'].median()

X_train['high_glazing'] = (X_train['glazing_area'] > median_glazing).astype(int)
X_test['high_glazing'] = (X_test['glazing_area'] > median_glazing).astype(int)

print(X_train['high_glazing'].value_counts())

high_glazing
0    528
1    240
Name: count, dtype: int64


In [9]:
#Check shapes
print("Updated X_train shape:", X_train.shape)
print("Updated X_test shape:", X_test.shape)

Updated X_train shape: (768, 13)
Updated X_test shape: (154, 13)


### 3. Encode Categorical Features
* One hot encode orientation and glazing_area_dist

In [10]:
# One hot encoded orientation and glazing area
X_train = pd.get_dummies(X_train, columns=['orientation', 'glazing_area_dist'], drop_first=True)
X_test = pd.get_dummies(X_test, columns=['orientation', 'glazing_area_dist'], drop_first=True)

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [11]:
#Check columns and shapes
print(X_train.select_dtypes(include='object').columns)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Index([], dtype='object')
X_train shape: (768, 19)
X_test shape: (154, 19)


# 4 Scale Numerical Freatures

# 5. Feature Selection

# 6. Save Final Data

In [12]:
X_train.to_csv('data/modeling/X_train.csv', index=False)
X_test.to_csv('data/modeling/X_test.csv', index=False)


print("Data saved!")
print(f"  X_Train.csv: {X_train.shape[1]} columns (scaled)")

Data saved!
  X_Train.csv: 19 columns (scaled)


### Summary
* Added 4 new features and one-hot encoded 2 existing features

##### Features Created:
* surface_wall_ratio
* height_surface_interaction
* log_cooling_load
* high_glazing

##### Processing Applied
* One-hot encoded: orientation and glazing_area_dist